# Configurable SeLU neural network for regression (fix)

This notebook provides one reusable PyTorch regression network for numerical arrays, EEG samples, and other tensor-shaped data. Each sample is flattened to a feature vector, so an input such as `(samples, channels, time)` works without changing the model code.

Change the `CONFIG` cell to control the input feature count, hidden layers, neurons per hidden layer, learning rate, maximum epochs, batch size, and early stopping. The final cell reports the model specifications, epochs actually run, and the MSE curve.

In [1]:
from dataclasses import dataclass, asdict
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

@dataclass
class NetworkConfig:
    # Set to an integer to enforce a specific number of input features.
    # None infers it from the flattened sample shape.
    input_features: int | None = None
    # The number of hidden layers is len(hidden_layers).
    # Each value is the number of neurons in that hidden layer.
    hidden_layers: tuple[int, ...] = (128, 64, 32)
    learning_rate: float = 1e-3
    max_epochs: int = 500
    batch_size: int = 64
    validation_split: float = 0.20
    patience: int = 60
    min_delta: float = 1e-7
    # Input standardization is recommended for SeLU/self-normalizing behavior.
    scale_inputs: bool = True
    # Target scaling can help when outputs have very different magnitudes.
    # MSE is always reported back in the original target units.
    scale_targets: bool = False
    dropout: float = 0.0
    device: str = "auto"
    random_seed: int = 42

CONFIG = NetworkConfig(
    input_features=None,       # e.g. 64; None = infer from X
    hidden_layers=(128, 64),   # e.g. () or (256, 128, 64)
    learning_rate=1e-3,
    max_epochs=500,
    batch_size=64,
    validation_split=0.20,
    patience=60,
    scale_inputs=True,
    scale_targets=False,
)

print(asdict(CONFIG))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\shrey\OneDrive\Desktop\Cosmos\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\shrey\OneDrive\Desktop\Cosmos\.venv\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "c:\Users\shrey\OneDrive\Desktop\Cosmos\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 807, in start
    self.io

{'input_features': None, 'hidden_layers': (128, 64), 'learning_rate': 0.001, 'max_epochs': 500, 'batch_size': 64, 'validation_split': 0.2, 'patience': 60, 'min_delta': 1e-07, 'scale_inputs': True, 'scale_targets': False, 'dropout': 0.0, 'device': 'auto', 'random_seed': 42}


In [2]:
def _to_numpy(value):
    """Convert NumPy, pandas, or PyTorch input to a CPU NumPy array."""
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    if hasattr(value, "to_numpy"):
        return value.to_numpy()
    return np.asarray(value)

def _flatten_samples(value, name):
    array = _to_numpy(value).astype(np.float32, copy=False)
    if array.ndim < 2:
        raise ValueError(f"{name} must have a sample dimension and at least one feature dimension.")
    flattened = array.reshape(array.shape[0], -1)
    if not np.isfinite(flattened).all():
        raise ValueError(f"{name} contains NaN or infinite values. Clean it before training.")
    return flattened

def _resolve_device(device_name):
    if device_name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device(device_name)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CONFIG.device is 'cuda', but CUDA is not available. Use 'auto' or 'cpu'.")
    return device

def _set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _make_model(input_features, output_features, config):
    if any(neurons <= 0 for neurons in config.hidden_layers):
        raise ValueError("Every hidden layer size must be a positive integer.")
    if not 0.0 <= config.dropout < 1.0:
        raise ValueError("dropout must be in the range [0, 1).")

    layers = []
    previous_features = input_features
    for neurons in config.hidden_layers:
        linear = nn.Linear(previous_features, neurons)
        # LeCun-normal initialization pairs naturally with SeLU.
        nn.init.normal_(linear.weight, mean=0.0, std=np.sqrt(1.0 / previous_features))
        nn.init.zeros_(linear.bias)
        layers.extend([linear, nn.SELU()])
        if config.dropout > 0:
            layers.append(nn.AlphaDropout(config.dropout))
        previous_features = neurons

    output = nn.Linear(previous_features, output_features)
    nn.init.xavier_uniform_(output.weight)
    nn.init.zeros_(output.bias)
    layers.append(output)
    return nn.Sequential(*layers)

def _inverse_targets(values, target_scaler):
    return target_scaler.inverse_transform(values) if target_scaler is not None else values

def _mse_in_original_units(model, x_tensor, y_tensor, target_scaler, device):
    with torch.no_grad():
        predictions = model(x_tensor.to(device)).cpu().numpy()
    actual = y_tensor.numpy()
    predictions = _inverse_targets(predictions, target_scaler)
    actual = _inverse_targets(actual, target_scaler)
    return float(np.mean((predictions - actual) ** 2))

def train_regression_network(X, y, config=CONFIG):
    """Train a configurable feed-forward SeLU network and return all artifacts."""
    _set_seed(config.random_seed)
    X_flat = _flatten_samples(X, "X")
    y_array = _to_numpy(y).astype(np.float32, copy=False)
    if y_array.ndim == 1:
        y_flat = y_array.reshape(-1, 1)
    else:
        y_flat = _flatten_samples(y_array, "y")
    if not np.isfinite(y_flat).all():
        raise ValueError("y contains NaN or infinite values. Clean it before training.")
    if X_flat.shape[0] != y_flat.shape[0]:
        raise ValueError("X and y must contain the same number of samples.")
    if X_flat.shape[0] < 4:
        raise ValueError("At least four samples are needed for a train/validation split.")
    if config.input_features is not None and X_flat.shape[1] != config.input_features:
        raise ValueError(
            f"X has {X_flat.shape[1]} flattened features, but CONFIG.input_features="
            f"{config.input_features}. Check the input shape or update CONFIG."
        )
    if not 0.0 <= config.validation_split < 1.0:
        raise ValueError("validation_split must be in the range [0, 1).")
    if config.max_epochs <= 0 or config.learning_rate <= 0:
        raise ValueError("max_epochs and learning_rate must be positive.")

    input_features = X_flat.shape[1]
    output_features = y_flat.shape[1]
    indices = np.arange(X_flat.shape[0])
    if config.validation_split > 0:
        train_idx, val_idx = train_test_split(
            indices, test_size=config.validation_split, random_state=config.random_seed
        )
    else:
        train_idx, val_idx = indices, np.array([], dtype=int)

    x_scaler = StandardScaler().fit(X_flat[train_idx]) if config.scale_inputs else None
    target_scaler = StandardScaler().fit(y_flat[train_idx]) if config.scale_targets else None

    def transform_inputs(data):
        return x_scaler.transform(data) if x_scaler is not None else data

    def transform_targets(data):
        return target_scaler.transform(data) if target_scaler is not None else data

    x_train = torch.from_numpy(transform_inputs(X_flat[train_idx]).astype(np.float32))
    y_train = torch.from_numpy(transform_targets(y_flat[train_idx]).astype(np.float32))
    x_val = torch.from_numpy(transform_inputs(X_flat[val_idx]).astype(np.float32)) if len(val_idx) else None
    y_val = torch.from_numpy(transform_targets(y_flat[val_idx]).astype(np.float32)) if len(val_idx) else None

    device = _resolve_device(config.device)
    model = _make_model(input_features, output_features, config).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    criterion = nn.MSELoss()
    train_loader = DataLoader(
        TensorDataset(x_train, y_train),
        batch_size=min(config.batch_size, len(x_train)),
        shuffle=True,
    )

    history = {"train_mse": [], "validation_mse": []}
    best_metric = np.inf
    best_epoch = 0
    best_state = deepcopy(model.state_dict())
    epochs_without_improvement = 0

    for epoch in range(1, config.max_epochs + 1):
        model.train()
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x_batch.to(device)), y_batch.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        model.eval()
        train_mse = _mse_in_original_units(model, x_train, y_train, target_scaler, device)
        validation_mse = (
            _mse_in_original_units(model, x_val, y_val, target_scaler, device)
            if x_val is not None else np.nan
        )
        history["train_mse"].append(train_mse)
        history["validation_mse"].append(validation_mse)

        metric = validation_mse if x_val is not None else train_mse
        if metric < best_metric - config.min_delta:
            best_metric = metric
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= config.patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    epochs_ran = len(history["train_mse"])
    specs = {
        "input_features": input_features,
        "original_input_shape": tuple(_to_numpy(X).shape[1:]),
        "output_features": output_features,
        "hidden_layers": tuple(config.hidden_layers),
        "activation": "SeLU",
        "optimizer": "Adam",
        "learning_rate": config.learning_rate,
        "max_epochs": config.max_epochs,
        "epochs_ran": epochs_ran,
        "best_epoch": best_epoch,
        "batch_size": config.batch_size,
        "validation_split": config.validation_split,
        "early_stopping_patience": config.patience,
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "device": str(device),
    }
    return {
        "model": model,
        "history": history,
        "specs": specs,
        "x_scaler": x_scaler,
        "target_scaler": target_scaler,
    }

## Provide data and train

Replace the demo `X` and `y` below with your own arrays. `X` can be `(samples, features)` or any tensor shape with samples first, such as `(samples, channels, time)`. `y` can be one-dimensional for a single regression target or multidimensional for multiple targets.

In [3]:
# Demo data: replace this block with your own X and y.
rng = np.random.default_rng(CONFIG.random_seed)
X = rng.normal(size=(600, 8)).astype(np.float32)
y = (
    2.5 * X[:, 0] - 1.7 * X[:, 1] ** 2 + 0.8 * X[:, 2] * X[:, 3]
    + rng.normal(scale=0.15, size=len(X))
).astype(np.float32)

# Example EEG/tensor input: X could instead have shape (samples, channels, time).
# The network will flatten each sample and infer input_features = channels * time.

result = train_regression_network(X, y, CONFIG)
model = result["model"]
model

RuntimeError: Numpy is not available

In [ ]:
def show_training_result(result):
    history = result["history"]
    specs = result["specs"]
    epochs = np.arange(1, len(history["train_mse"]) + 1)

    print("Model specifications")
    for key, value in specs.items():
        print(f"{key}: {value}")
    validation_values = np.asarray(history["validation_mse"], dtype=float)
    if np.isnan(validation_values).all():
        print("validation MSE: not used (validation_split=0)")
    else:
        print(f"best validation MSE: {np.nanmin(validation_values):.6g}")
    print(f"final training MSE: {history['train_mse'][-1]:.6g}")

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(epochs, history["train_mse"], label="Training MSE", linewidth=2)
    if not np.isnan(history["validation_mse"]).all():
        ax.plot(epochs, history["validation_mse"], label="Validation MSE", linewidth=2)
        ax.axvline(specs["best_epoch"], color="gray", linestyle="--", alpha=0.7, label="Best epoch")
    ax.set_title("SeLU neural network: MSE over training")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Mean squared error (original target units)")
    ax.grid(True, alpha=0.25)
    ax.legend()

    spec_text = (
        f"Input: {specs['input_features']} features | Output: {specs['output_features']}\n"
        f"Hidden: {specs['hidden_layers']} | Activation: {specs['activation']}\n"
        f"LR: {specs['learning_rate']} | Max epochs: {specs['max_epochs']} | Ran: {specs['epochs_ran']}\n"
        f"Best epoch: {specs['best_epoch']} | Parameters: {specs['parameters']} | Device: {specs['device']}"
    )
    fig.text(0.5, 0.01, spec_text, ha="center", va="bottom", fontsize=9)
    plt.tight_layout(rect=(0, 0.10, 1, 1))
    plt.show()

show_training_result(result)

In [ ]:
def predict(result, X_new):
    """Predict on new samples with the same flattening and input scaling."""
    specs = result["specs"]
    x_new = _flatten_samples(X_new, "X_new")
    if x_new.shape[1] != specs["input_features"]:
        raise ValueError(f"X_new must flatten to {specs['input_features']} features.")
    if result["x_scaler"] is not None:
        x_new = result["x_scaler"].transform(x_new)
    x_tensor = torch.from_numpy(x_new.astype(np.float32))
    device = next(result["model"].parameters()).device
    with torch.no_grad():
        prediction = result["model"](x_tensor.to(device)).cpu().numpy()
    prediction = _inverse_targets(prediction, result["target_scaler"])
    return prediction[:, 0] if prediction.shape[1] == 1 else prediction

# Example: predictions = predict(result, X[:5])